In [ ]:
import torch
import torch.nn as nn
from torch.optim import Adam
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
from torchvision import models
from sklearn.preprocessing import LabelEncoder
import pandas as pd
from PIL import Image
import os
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
train_df = pd.read_csv('train/train.csv')
val_df = pd.read_csv('train/val.csv')

In [ ]:
train_df["category"].unique()

In [ ]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.ConvertImageDtype(torch.float)
])

In [ ]:
class CustomImageDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform
        self.labels = torch.tensor(dataframe["category"]).to(device)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_path = self.dataframe.iloc[idx,0]
        label = self.labels[idx]
        image = Image.open(img_path)

        if self.transform:
            image = (self.transform(image)/255.0).to(device)
        return image, label

In [ ]:
train_dataset = CustomImageDataset(train_df, transform=transform)
val_dataset = CustomImageDataset(val_df, transform=transform)

In [ ]:
n_rows = 3
n_cols = 3

f, axarr = plt.subplots(n_rows,n_cols)

for row in range(n_rows):
    for col in range(n_cols):
        image = train_dataset[np.random.randint(0, len(train_dataset))][0].detach().cpu()
        image = image.squeeze(0) if image.shape[0] == 1 else image.permute(1, 2, 0)
        axarr[row, col].imshow((image * 255.0).numpy())
        axarr[row, col].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
LR = 1e-3
BATCH_SIZE = 4
EPOCH = 15

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
googlenet_model = models.googlenet(weights="DEFAULT")

In [ ]:

for param in googlenet_model.parameters():
    param.requires_grad = True
    

In [ ]:
googlenet_model.fc

In [ ]:
num_classes = len(train_df["category"].unique())

num_classes

In [ ]:
googlenet_model.fc = torch.nn.Linear(googlenet_model.fc.in_features, num_classes)
googlenet_model.fc

In [ ]:
googlenet_model.to(device)

In [ ]:
loss_fun = nn.CrossEntropyLoss()
optimizer = Adam(googlenet_model.parameters(), lr=LR)

total_loss_train_plot = []
total_acc_train_plot = []


for epoch in range(EPOCH):
    total_acc_train = 0
    total_loss_train= 0

    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = googlenet_model(inputs)
        train_loss = loss_fun(outputs, labels)
        total_loss_train+=train_loss.item()

        train_loss.backward()

        train_acc = (torch.argmax(outputs,axis=1)==labels).sum().item()
        total_acc_train+=train_acc

        optimizer.step()


    total_loss_train_plot.append(round(total_loss_train/1000,4))
    total_acc_train_plot.append(round(total_acc_train/train_dataset.__len__()*100,4))

    print(f"Epoch {epoch+1}/{EPOCH}, Train Loss : {round(total_loss_train/1000,4)}, Train Accuracy : {round(total_acc_train/train_dataset.__len__()*100,4)}")
    
    


In [ ]:
with torch.no_grad():

    total_loss_test = 0

    total_acc_test = 0



    for inputs, labels in val_loader:

        prediction = googlenet_model(inputs)



        test_acc = (torch.argmax(prediction,axis=1)==labels).sum().item()

        total_acc_test += test_acc

In [ ]:
print("total_loss_test:-",(round(total_loss_test/1000,4)))
print("total_acc_test:-",{round(total_acc_test/val_dataset.__len__()*100,4)})


Transfer Learning

In [ ]:
googlenet_model = models.googlenet(weights="DEFAULT")



for param in googlenet_model.parameters():
    param.requires_grad = False


googlenet_model.fc = torch.nn.Linear(googlenet_model.fc.in_features, num_classes)
googlenet_model.fc.requires_grad = True

googlenet_model.to(device)

In [ ]:
loss_fun = nn.CrossEntropyLoss()
optimizer = Adam(googlenet_model.parameters(), lr=LR)

total_loss_train_plot = []
total_acc_train_plot = []


for epoch in range(EPOCH):
    total_acc_train = 0
    total_loss_train= 0

    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = googlenet_model(inputs)
        train_loss = loss_fun(outputs, labels)
        total_loss_train+=train_loss.item()

        train_loss.backward()

        train_acc = (torch.argmax(outputs,axis=1)==labels).sum().item()
        total_acc_train+=train_acc

        optimizer.step()


    total_loss_train_plot.append(round(total_loss_train/1000,4))
    total_acc_train_plot.append(round(total_acc_train/train_dataset.__len__()*100,4))

    print(f"Epoch {epoch+1}/{EPOCH}, Train Loss : {round(total_loss_train/1000,4)}, Train Accuracy : {round(total_acc_train/train_dataset.__len__()*100,4)}")
    
    


In [ ]:
with torch.no_grad():

    total_loss_test = 0

    total_acc_test = 0



    for inputs, labels in val_loader:

        prediction = googlenet_model(inputs)



        test_acc = (torch.argmax(prediction,axis=1)==labels).sum().item()

        total_acc_test += test_acc


print("total_acc_test:-",round(total_acc_test/val_dataset.__len__()*100,4))